# Laplace Direct Nearfield Experiments

Experiments:

1. Linear geometry, increasing trace order, fixed nearfield target. This isolates p-version trace approximation and the high-order density effect in nearfield evaluation.
2. Linear geometry, lowest conforming trace orders, target approaches the surface. This compares corrected nearfield evaluation with naive Gauss quadrature.
3. Fixed low-order trace spaces, increasing geometry order, fixed target. This isolates the influence of curved elements.
4. Isoparametric sequence: geometry order and H1/2 order are equal, while the H-1/2 order is one less.


In [ ]:
import time
from dataclasses import dataclass
from pathlib import Path
from typing import Iterable

import matplotlib.pyplot as plt
import ngsolve as ngs
import numpy as np
from netgen.occ import Box, OCCGeometry, Pnt as OCCPnt, Sphere
from ngsolve import BND, GridFunction, H1, Integrate, Mesh, SurfaceL2, TaskManager, ds, sqrt, x, y, z
from ngsolve.bem import LaplaceDL, LaplaceSL


FOUR_PI = 4.0 * np.pi


@dataclass(frozen=True)
class TraceCase:
    experiment: str
    maxh: float
    curve_order: int
    hhalf_order: int
    hminus_order: int
    distance: float
    quadrature: str
    bonus_intorder: int
    naive_intorder: int


def as_tuple(a) -> tuple[float, float, float]:
    return tuple(float(component) for component in a)


def normalize(a: tuple[float, float, float]) -> tuple[float, float, float]:
    a = np.asarray(a, dtype=float)
    length = float(np.linalg.norm(a))
    if length == 0.0:
        raise ValueError('cannot normalize zero vector')
    return as_tuple(a / length)


def default_target_direction(source_point: tuple[float, float, float]) -> tuple[float, float, float]:
    if np.linalg.norm(np.asarray(source_point, dtype=float)) == 0.0:
        return (-1.0, 0.0, 0.0)
    return as_tuple(-np.asarray(normalize(source_point)))


def exact_fundamental_cf(source_point: tuple[float, float, float]):
    sx, sy, sz = source_point
    return 1.0 / (FOUR_PI * sqrt((x - sx) ** 2 + (y - sy) ** 2 + (z - sz) ** 2))


def exact_neumann_trace_cf(source_point: tuple[float, float, float]):
    sx, sy, sz = source_point
    normal = ngs.specialcf.normal(3)
    r = sqrt((x - sx) ** 2 + (y - sy) ** 2 + (z - sz) ** 2)
    radial_normal = (x - sx) * normal[0] + (y - sy) * normal[1] + (z - sz) * normal[2]
    return -radial_normal / (FOUR_PI * r ** 3)


def make_sphere_mesh(radius: float, maxh: float, curve_order: int) -> Mesh:
    sphere = Sphere((0, 0, 0), radius)
    sphere.faces.name = 'sphere'
    mesh = Mesh(OCCGeometry(sphere).GenerateMesh(maxh=maxh, quad_dominated=False))
    return mesh.Curve(curve_order)


def boundary_count(mesh: Mesh) -> int:
    return sum(1 for _ in mesh.Elements(BND))


def make_point_evaluation_mesh(
    point: tuple[float, float, float],
    size: float,
) -> Mesh:
    point = np.asarray(point, dtype=float)
    lower = point - size
    upper = point + size
    box = Box(OCCPnt(*lower), OCCPnt(*upper))
    return Mesh(OCCGeometry(box).GenerateMesh(maxh=size))


def target_point(direction: tuple[float, float, float], distance: float) -> tuple[float, float, float]:
    direction = np.asarray(normalize(direction), dtype=float)
    return as_tuple((1.0 + distance) * direction)


def target_mesh_size_for(distance: float) -> float:
    return max(0.1 * distance, 1e-5)


def geometry_errors(mesh: Mesh, radius: float, order: int) -> tuple[float, float, float]:
    region = mesh.Boundaries('sphere')
    r = sqrt(x * x + y * y + z * z)
    radial_error = r - radius
    area = Integrate(1, mesh, definedon=region, order=order)
    rms = sqrt(Integrate(radial_error * radial_error, mesh, definedon=region, order=order) / area)

    element_errors = Integrate(
        sqrt(radial_error * radial_error),
        mesh,
        definedon=region,
        element_wise=True,
        order=order,
    )
    element_areas = Integrate(1, mesh, definedon=region, element_wise=True, order=order)
    max_mean = 0.0
    for error, element_area in zip(element_errors, element_areas):
        max_mean = max(max_mean, float(error / element_area))

    return float(area), float(rms), max_mean


def project_cauchy_data(
    mesh: Mesh,
    hhalf_order: int,
    hminus_order: int,
    source_point: tuple[float, float, float],
):
    trace_space = H1(mesh, order=hhalf_order)
    flux_space = SurfaceL2(mesh, order=hminus_order, dual_mapping=True)

    trace = GridFunction(trace_space)
    flux = GridFunction(flux_space)
    region = mesh.Boundaries('sphere')

    with TaskManager():
        trace.Set(exact_fundamental_cf(source_point), definedon=region)
        flux.Set(exact_neumann_trace_cf(source_point), definedon=region)

    return trace_space, trace, flux_space, flux


def evaluate_corrected_direct(
    mesh: Mesh,
    trace_space,
    trace: GridFunction,
    flux_space,
    flux: GridFunction,
    target_mesh_point,
    bonus_intorder: int,
    bem_params: dict[str, object],
) -> float:
    trace_trial = trace_space.TrialFunction()
    flux_trial = flux_space.TrialFunction()
    with TaskManager():
        sl = LaplaceSL(flux_trial * ds(bonus_intorder=bonus_intorder), **bem_params)(flux)
        dl = LaplaceDL(trace_trial * ds(bonus_intorder=bonus_intorder), **bem_params)(trace)
        value = (dl - sl)(target_mesh_point)

    return float(value.real if hasattr(value, 'real') else value)


def evaluate_naive_gauss_direct(
    mesh: Mesh,
    trace: GridFunction,
    flux: GridFunction,
    point: tuple[float, float, float],
    intorder: int,
) -> float:
    tx, ty, tz = point
    dx = tx - x
    dy = ty - y
    dz = tz - z
    r = sqrt(dx * dx + dy * dy + dz * dz)
    normal = ngs.specialcf.normal(3)

    sl_kernel = 1.0 / (FOUR_PI * r)
    dl_kernel = (normal[0] * dx + normal[1] * dy + normal[2] * dz) / (FOUR_PI * r ** 3)
    integrand = dl_kernel * trace - sl_kernel * flux

    with TaskManager():
        value = Integrate(integrand, mesh, definedon=mesh.Boundaries('sphere'), order=intorder)

    return float(value.real if hasattr(value, 'real') else value)


def exact_point_value(
    source_point: tuple[float, float, float],
    point: tuple[float, float, float],
) -> float:
    distance = np.linalg.norm(np.asarray(point, dtype=float) - np.asarray(source_point, dtype=float))
    return float(1.0 / (FOUR_PI * distance))


def run_trace_case(
    case: TraceCase,
    source_point: tuple[float, float, float],
    direction: tuple[float, float, float],
    bem_params: dict[str, object],
) -> dict[str, object]:
    mesh = make_sphere_mesh(1.0, case.maxh, case.curve_order)
    trace_space, trace, flux_space, flux = project_cauchy_data(
        mesh,
        case.hhalf_order,
        case.hminus_order,
        source_point,
    )

    point = target_point(direction, case.distance)
    target_mesh_size = target_mesh_size_for(case.distance)
    target_mesh = make_point_evaluation_mesh(point, target_mesh_size)
    target_mesh_point = target_mesh(*point)
    start = time.perf_counter()
    if case.quadrature == 'corrected':
        value = evaluate_corrected_direct(
            mesh,
            trace_space,
            trace,
            flux_space,
            flux,
            target_mesh_point,
            case.bonus_intorder,
            bem_params,
        )
    elif case.quadrature == 'naive_gauss':
        value = evaluate_naive_gauss_direct(mesh, trace, flux, point, case.naive_intorder)
    else:
        raise ValueError(f'unknown quadrature: {case.quadrature}')
    elapsed = time.perf_counter() - start

    exact = exact_point_value(source_point, point)
    abs_error = abs(value - exact)
    rel_error = abs_error / abs(exact)
    geometry_order = max(2 * case.curve_order + 6, 8)
    source_area, source_geom_rms, source_geom_max = geometry_errors(mesh, 1.0, geometry_order)

    return {
        'experiment': case.experiment,
        'maxh': case.maxh,
        'curve_order': case.curve_order,
        'hhalf_order': case.hhalf_order,
        'hminus_order': case.hminus_order,
        'distance': case.distance,
        'quadrature': case.quadrature,
        'bonus_intorder': case.bonus_intorder,
        'naive_intorder': case.naive_intorder,
        'source_elements': boundary_count(mesh),
        'hhalf_ndof': trace_space.ndof,
        'hminus_ndof': flux_space.ndof,
        'total_ndof': trace_space.ndof + flux_space.ndof,
        'source_area': source_area,
        'source_geom_rms': source_geom_rms,
        'source_geom_max_el_abs': source_geom_max,
        'target_exact_distance': float(np.linalg.norm(point) - 1.0),
        'target_mesh_size': target_mesh_size,
        'target_x': point[0],
        'target_y': point[1],
        'target_z': point[2],
        'exact_value': exact,
        'value': value,
        'abs_error': abs_error,
        'rel_error': rel_error,
        'eval_seconds': elapsed,
    }


def run_cases(cases: Iterable[TraceCase], source_point, direction, bem_params):
    rows = []
    for case in cases:
        row = run_trace_case(case, source_point, direction, bem_params)
        rows.append(row)
        print(
            f"{case.experiment}: curve={case.curve_order}, "
            f"H1/2={case.hhalf_order}, H-1/2={case.hminus_order}, "
            f"distance={case.distance:g}, {case.quadrature}, "
            f"rel_error={row['rel_error']:.3e}"
        )
    return rows


def as_dataframe(rows):
    try:
        import pandas as pd
        try:
            display
        except NameError:
            def display(value):
                print(value)
        df = pd.DataFrame(rows)
        display(df)
        return df
    except ImportError:
        for row in rows:
            print(row)
        return None


In [ ]:
source_point = (0.25, -0.20, 0.15)
target_direction = default_target_direction(source_point)

maxh = 0.4
bem_params = {"use_fmm": False}
bonus_intorder = 8
naive_intorder = 8
fixed_distance = 0.0003
near_distances = [0.1, 0.03, 0.01, 0.003, 0.001, 0.0003]


## Experiment 1: p-version on linear geometry

Geometry is fixed to `Curve(1)`. The target point is fixed in the nearfield. The H1/2 order is increased, and the H-1/2 order is chosen one degree lower. This tests the trace approximation and the high-order density behavior of the corrected nearfield evaluation without a BEM solve.


In [ ]:
experiment_1_cases = [
    TraceCase(
        experiment='exp1_p_linear_geometry',
        maxh=maxh,
        curve_order=1,
        hhalf_order=p,
        hminus_order=max(p - 1, 0),
        distance=fixed_distance,
        quadrature='corrected',
        bonus_intorder=bonus_intorder,
        naive_intorder=naive_intorder,
    )
    for p in [1, 2, 3, 4, 5]
]

experiment_1_rows = run_cases(experiment_1_cases, source_point, target_direction, bem_params)
experiment_1_df = as_dataframe(experiment_1_rows)


In [ ]:
fig, ax = plt.subplots(figsize=(6, 4))
orders = [row['hhalf_order'] for row in experiment_1_rows]
errors = [row['rel_error'] for row in experiment_1_rows]
ax.semilogy(orders, errors, 'o-')
ax.set_xlabel('H1/2 order')
ax.set_ylabel('Relative error')
ax.set_title('Experiment 1: p-version, linear geometry')
ax.set_xticks(orders)
ax.grid(True, which='both', linestyle=':', linewidth=0.8)
fig.tight_layout()


## Experiment 2: corrected nearfield vs naive Gauss

Geometry is linear and the trace spaces are fixed to the lowest conforming pair: H1/2 order 1 and H-1/2 order 0. The target point approaches the exact sphere along a fixed exterior ray. This isolates the near-singular quadrature behavior.


In [ ]:
experiment_2_cases = []
for distance in near_distances:
    for quadrature in ['corrected', 'naive_gauss']:
        experiment_2_cases.append(
            TraceCase(
                experiment='exp2_corrected_vs_naive',
                maxh=maxh,
                curve_order=1,
                hhalf_order=1,
                hminus_order=0,
                distance=distance,
                quadrature=quadrature,
                bonus_intorder=bonus_intorder,
                naive_intorder=naive_intorder,
            )
        )

experiment_2_rows = run_cases(experiment_2_cases, source_point, target_direction, bem_params)
experiment_2_df = as_dataframe(experiment_2_rows)


In [ ]:
fig, ax = plt.subplots(figsize=(6, 4))
for quadrature, marker in [('corrected', 'o-'), ('naive_gauss', 's--')]:
    subset = sorted(
        [row for row in experiment_2_rows if row['quadrature'] == quadrature],
        key=lambda row: row['distance'],
    )
    ax.loglog(
        [row['distance'] for row in subset],
        [row['rel_error'] for row in subset],
        marker,
        label=quadrature,
    )
ax.invert_xaxis()
ax.set_xlabel('Distance to exact sphere')
ax.set_ylabel('Relative error')
ax.set_title('Experiment 2: target approaches surface')
ax.grid(True, which='both', linestyle=':', linewidth=0.8)
ax.legend()
fig.tight_layout()


## Experiment 3: geometry order with fixed low-order traces

The trace spaces are fixed to H1/2 order 1 and H-1/2 order 0. The target point is fixed. Only the geometry order is increased. This isolates the influence of curved elements when the trace approximation is kept low order.


In [ ]:
experiment_3_cases = [
    TraceCase(
        experiment='exp3_geometry_order_low_trace',
        maxh=maxh,
        curve_order=curve_order,
        hhalf_order=1,
        hminus_order=0,
        distance=fixed_distance,
        quadrature='corrected',
        bonus_intorder=bonus_intorder,
        naive_intorder=naive_intorder,
    )
    for curve_order in [1, 2, 3, 4]
]

experiment_3_rows = run_cases(experiment_3_cases, source_point, target_direction, bem_params)
experiment_3_df = as_dataframe(experiment_3_rows)


In [ ]:
fig, ax = plt.subplots(figsize=(6, 4))
orders = [row['curve_order'] for row in experiment_3_rows]
errors = [row['rel_error'] for row in experiment_3_rows]
geom = [row['source_geom_rms'] for row in experiment_3_rows]
ax.semilogy(orders, errors, 'o-', label='potential error')
ax.semilogy(orders, geom, '^--', label='geometry RMS')
ax.set_xlabel('Geometry order')
ax.set_ylabel('Relative error / geometry RMS')
ax.set_title('Experiment 3: geometry order, low trace order')
ax.set_xticks(orders)
ax.grid(True, which='both', linestyle=':', linewidth=0.8)
ax.legend()
fig.tight_layout()


## Experiment 4: isoparametric sequence

The geometry order and the H1/2 order are equal. The H-1/2 order is one lower. The target point is fixed. This is the isoparametric trace/geometry sequence relevant for the paper.


In [ ]:
experiment_4_cases = [
    TraceCase(
        experiment='exp4_isoparametric',
        maxh=maxh,
        curve_order=p,
        hhalf_order=p,
        hminus_order=max(p - 1, 0),
        distance=fixed_distance,
        quadrature='corrected',
        bonus_intorder=bonus_intorder,
        naive_intorder=naive_intorder,
    )
    for p in [1, 2, 3, 4]
]

experiment_4_rows = run_cases(experiment_4_cases, source_point, target_direction, bem_params)
experiment_4_df = as_dataframe(experiment_4_rows)


In [ ]:
fig, ax = plt.subplots(figsize=(6, 4))
orders = [row['curve_order'] for row in experiment_4_rows]
errors = [row['rel_error'] for row in experiment_4_rows]
geom = [row['source_geom_rms'] for row in experiment_4_rows]
ax.semilogy(orders, errors, 'o-', label='potential error')
ax.semilogy(orders, geom, '^--', label='geometry RMS')
ax.set_xlabel('Isoparametric order')
ax.set_ylabel('Relative error / geometry RMS')
ax.set_title('Experiment 4: isoparametric sequence')
ax.set_xticks(orders)
ax.grid(True, which='both', linestyle=':', linewidth=0.8)
ax.legend()
fig.tight_layout()


## Combined data export


In [ ]:
all_rows = experiment_1_rows + experiment_2_rows + experiment_3_rows + experiment_4_rows
all_df = as_dataframe(all_rows)

csv_path = Path.cwd() / 'laplace_direct_nearfield_experiments.csv'
if all_df is not None:
    all_df.to_csv(csv_path, index=False)
    print(csv_path)
